# Comparison of *in situ* observsations and the ERA5 reanalysis climatology

[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/c3s-training-submodule-insitu-obs/main?labpath=insitu-obs-against-climatology.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/c3s-training-submodule-insitu-obs/blob/main/insitu-obs-against-climatology.ipynb)
[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/c3s-training-submodule-insitu-obs/blob/main/insitu-obs-against-climatology.ipynb)

:::{note}
This notebook can be run on free online platforms, such as Binder, Kaggle and Colab, or they can be accessed from GitHub. The links to run this notebook in these environments are provided here, but please note they are not supported by ECMWF.
:::

*Provide an introduction and context for the notebook here.*


## Learning objectives 🎯


This notebook will teach you how to:
1. Access data from the Climate Data Store using `earthkit-data`
2. Calculate climatological averages using `earthkit-transforms`
3. Create publication quality figures comparing the data sources using `earthkit-plots`


## Prepare your environment


*Insert here any necessary instructions to set-up the environment of learners, any background information on data, projects, access to data catalogues, or preliminary steps necessary to run the notebooks. These may include instructions for installing packages, imports, etc.*

:::{important}
All required dependencies should be included in the `environment.yml` file.
:::

### Import libraries


In [1]:
# Import your libraries here
import earthkit.data as ekd
import earthkit.transforms as ekt
import earthkit.plots as ekp

### Select a region and period of interest

For this example demonstration, we will look at the
[Great Storm of 1987](https://en.wikipedia.org/wiki/Great_storm_of_1987).
The storm occured on the night of the 15th/16th October, we will look at the
antecedent build up of meteorological conditions and compare the observations
with the long-term climatological behaviour.

In [2]:
# Period and region of interest
area = [65, -10, 45, 30]  # North, West, South, East
year = "1987"
month = "10"
days = ["14", "15", "16", "17"]

## Download data from the CDS

Earthkit-data uses your CDS-API credentials to download data, it is advised that you set up
a `~/.cdsapirc` file with your credentials following the
[how to api instructions](https://cds.climate.copernicus.eu/how-to-api).
If you have not setup your `~/.cdsapirc` file, you will be prompted for your credentials
when executing the cells below.

### ERA5 monthly data for climatology

We will use the 

In [3]:
era5_dataset = "reanalysis-era5-single-levels-monthly-means"
variables = [
    "2m_temperature",
    "total_precipitation",
    "sea_surface_temperature",
    # "10m_u_component_of_wind",
    # "10m_v_component_of_wind",
    # "2m_dewpoint_temperature",
    # "mean_sea_level_pressure",
    # "mean_wave_direction",
    # "mean_wave_period",
    # "significant_height_of_combined_wind_waves_and_swell",
    # "surface_pressure",
]
base_request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "year": [f"{year:04d}" for year in range(1961, 1990)],
    "month": [f"{month:02d}" for month in range(1, 13)],
    "time": ["00:00"],
    "area": area,
}

era5_datasets = {}
for var in variables:
    request = base_request.copy()
    request["variable"] = var
    era5_ekds = ekd.from_source("cds", era5_dataset, request)
    era5_datasets[var] = era5_ekds.to_xarray(time_dim_mode="valid_time")

# Update the Accumulated variables to use the same dimension as the other variables for easier comparison
ref_var = "2m_temperature"
for var in ["total_precipitation"]:
    era5_datasets[var] = era5_datasets[var].assign_coords({"valid_time": era5_datasets[ref_var].valid_time})

# Now we can merge the datasets into a single xarray Dataset
import xarray as xr
era5_ds = xr.merge(era5_datasets.values())
era5_ds

<xarray.Dataset> Size: 109MB
Dimensions:     (valid_time: 348, latitude: 81, longitude: 161)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 3kB 1961-01-01 ... 1989-12-01
  * latitude    (latitude) float64 648B 65.0 64.75 64.5 ... 45.5 45.25 45.0
  * longitude   (longitude) float64 1kB -10.0 -9.75 -9.5 ... 29.5 29.75 30.0
Data variables:
    2t          (valid_time, latitude, longitude) float64 36MB ...
    tp          (valid_time, latitude, longitude) float64 36MB ...
    sst         (valid_time, latitude, longitude) float64 36MB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [4]:
era5_climatologies = ekt.climatology.monthly_mean(era5_ds)
era5_climatologies

<xarray.Dataset> Size: 4MB
Dimensions:    (month: 12, latitude: 81, longitude: 161)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 648B 65.0 64.75 64.5 64.25 ... 45.5 45.25 45.0
  * longitude  (longitude) float64 1kB -10.0 -9.75 -9.5 ... 29.5 29.75 30.0
Data variables:
    2t         (month, latitude, longitude) float64 1MB 274.9 274.9 ... 278.5
    tp         (month, latitude, longitude) float64 1MB 0.002677 ... 0.00177
    sst        (month, latitude, longitude) float64 1MB 276.3 276.2 ... 281.8
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

### ERA5 Hourly data

Select the hourly data from ERA5 single levels.

In [6]:
era5_hourly_dataset = "reanalysis-era5-single-levels"
variables = [
    "2m_temperature",
    "total_precipitation",
    "sea_surface_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",
    "mean_sea_level_pressure",
    "mean_wave_direction",
    "mean_wave_period",
    "significant_height_of_combined_wind_waves_and_swell",
    "surface_pressure",
]
era5_hourly_request = {
    "product_type": ["reanalysis"],
    "variable": variables,
    "year": year,
    "month": month,
    "day": days,
    "time": ["00:00"],
    "area": area,
}

era5_hourly_ekds = ekd.from_source("cds", era5_hourly_dataset, era5_hourly_request)
era5_hourly_ds = era5_hourly_ekds.to_xarray(time_dim_mode="valid_time")
era5_hourly_ds

2026-04-13 16:18:03,679 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-13 16:18:03,680 INFO Request ID is e8f37595-b076-46f5-9087-a009b0adbf0f
2026-04-13 16:18:03,758 INFO status has been updated to accepted
2026-04-13 16:18:25,069 INFO status has been updated to running
2026-04-13 16:18:36,530 INFO status has been updated to successful


76af9c13274c11e111e9e6587adc662f.grib:   0%|          | 0.00/802k [00:00<?, ?B/s]

<xarray.Dataset> Size: 5MB
Dimensions:     (valid_time: 4, latitude: 81, longitude: 161)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 32B 1987-10-14 ... 1987-10-17
  * latitude    (latitude) float64 648B 65.0 64.75 64.5 ... 45.5 45.25 45.0
  * longitude   (longitude) float64 1kB -10.0 -9.75 -9.5 ... 29.5 29.75 30.0
Data variables:
    10u         (valid_time, latitude, longitude) float64 417kB ...
    10v         (valid_time, latitude, longitude) float64 417kB ...
    2d          (valid_time, latitude, longitude) float64 417kB ...
    2t          (valid_time, latitude, longitude) float64 417kB ...
    msl         (valid_time, latitude, longitude) float64 417kB ...
    mwd         (valid_time, latitude, longitude) float64 417kB ...
    mwp         (valid_time, latitude, longitude) float64 417kB ...
    sp          (valid_time, latitude, longitude) float64 417kB ...
    sst         (valid_time, latitude, longitude) float64 417kB ...
    swh         (valid_time, latitude, longitude) float64 417kB ...
    tp          (valid_time, latitude, longitude) float64 417kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

### *in situ* data

Select the 

In [ ]:
dataset = "insitu-observations-surface-marine"
request = {
    "version": "2_0_0",
    "variable": [
        "air_pressure_at_sea_level",
        "air_temperature",
        "dew_point_temperature",
        "water_temperature",
        "wind_from_direction",
        "wind_speed"
    ],
    "year": year,
    "month": month,
    "day": days,
    "area": area,
}
insitu_ekds = ekd.from_source("cds", dataset, request)
insitu_ds = insitu_ekds.to_xarray()
insitu_ds["observed_variable"] = insitu_ds.observed_variable.str.decode("UTF-8")
# Assign some of the data variables as coordinates for easier selection and plotting
for coord in ["latitude", "longitude", "report_timestamp", "observed_variable", "observation_height_above_station_surface"]:
    insitu_ds = insitu_ds.assign_coords({coord: insitu_ds[coord]})
unique_observed_variables = insitu_ds.observed_variable.to_index().unique()

insitu_ds

2026-04-13 16:20:15,450 INFO Request ID is f83adce5-5caf-4410-9176-4be0e096ab32
2026-04-13 16:20:15,516 INFO status has been updated to accepted


### Storm Track data

The CDS also offers Storm Track data which can be used to map storms

In [ ]:
st_dataset = "sis-european-wind-storm-reanalysis"
st_request = {
    "product": "windstorm_track",
    "variable": "all",
    "tracking_algorithm": ["hodges"],
    "event_aggregation": "single_event",
    "year": ["1987"],
    "month": ["10"],
    "day": ["15"]
}
st_ekds = ekd.from_source("cds", st_dataset, st_request)
st_ds = st_ekds.to_xarray()
st_ds = st_ds.assign_coords({
    "latitude": st_ds.latitude, "longitude": st_ds.longitude, "time": st_ds.time.astype("datetime64[ns]")
})
st_ds

<xarray.Dataset> Size: 864B
Dimensions:    (index: 12)
Coordinates:
  * index      (index) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    time       (index) datetime64[ns] 96B 1987-10-15 ... 1987-10-17T18:00:00
    latitude   (index) float64 96B 42.25 42.5 44.25 46.5 ... 66.25 68.0 69.75
    longitude  (index) float64 96B -23.25 -17.5 -13.5 -9.0 ... 0.5 1.5 5.0 6.75
Data variables:
    id         (index) int64 96B 1014 1014 1014 1014 ... 1014 1014 1014 1014
    fg10       (index) float64 96B 12.15 5.343 16.13 16.74 ... 12.07 6.627 7.068
    lsm        (index) float64 96B 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    msl        (index) float64 96B 9.86e+04 9.791e+04 ... 9.741e+04 9.778e+04
    algorithm  (index) object 96B 'hodges' 'hodges' ... 'hodges' 'hodges'

## Take home messages 📌


*In this section, summarise key take home messages.*

- *Key message 1*
- *Key message 2*
- *Key message 3*
